In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import numpy as np
import torch
from torch import nn, optim, autograd
from math import pi
from sklearn.model_selection import RepeatedKFold


torch.manual_seed(123456)
np.random.seed(123456)

class Unit(nn.Module):
    def __init__(self, in_N, out_N):
        super(Unit, self).__init__()
        self.in_N = in_N
        self.out_N = out_N
        self.L = nn.Linear(in_N, out_N)

    def forward(self, x):
        x1 = self.L(x)
        x2 = torch.tanh(x1)
        return x2

class Unit1(nn.Module):
    def __init__(self, in_N, out_N):
        super(Unit1, self).__init__()
        self.in_N = in_N
        self.out_N = out_N
        self.L = nn.Linear(in_N, out_N)

    def forward(self, x):
        x1 = self.L(x)
        x2 = torch.sigmoid(x1)
        return x2


class NN1(nn.Module): #Nonlinear component
    def __init__(self, in_N, width, depth, out_N):
        super(NN1, self).__init__()
        self.width = width
        self.in_N = in_N
        self.out_N = out_N
        self.stack = nn.ModuleList()
        self.stack.append(Unit(in_N, width[0]))
        for i in range(1,depth-1):
            self.stack.append(Unit(width[i-1], width[i]))
        self.stack.append(Unit1(width[depth-2], width[depth-1])) #nn.Linear

    def forward(self, x):
        for i in range(len(self.stack)):
            x = self.stack[i](x)
        return x

    
class NN2(nn.Module): #Linear component
    def __init__(self, in_N, width, depth, out_N):
        super(NN2, self).__init__()
        self.in_N = in_N
        self.width = width
        self.depth = depth
        self.out_N = out_N
        self.stack = nn.ModuleList()
        self.stack.append(nn.Linear(in_N, width[0]))
        for i in range(1,depth-1):
            self.stack.append(nn.Linear(width[i-1], width[i]))    
        self.stack.append(Unit1(width[depth-2], width[depth-1])) #nn.Linear

    def forward(self, x):
        for i in range(len(self.stack)):
            x = self.stack[i](x)
        return x


def weights_init(m):
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        nn.init.xavier_normal_(m.weight)
        nn.init.constant_(m.bias, 0.0)


class Unit3(nn.Module):
    def __init__(self, in_N, out_N,actf):
        super(Unit3, self).__init__()
        self.in_N = in_N
        self.out_N = out_N
        self.actf = actf
        self.L = nn.Linear(in_N, out_N)

    def forward(self, x):
        actf=self.actf
        x1 = self.L(x)
        if actf==0:
            x2 = torch.tanh(x1)
        elif actf==1:
            x2 = torch.sigmoid(x1) 
        elif actf==2:
            x2 = torch.relu(x1)
        elif actf==3:
            x2 = torch.selu(x1)
        return x2
    
class NN3(nn.Module):
    def __init__(self, in_N, width1, depth1,width2, depth2,out_N,bn,dp,dprate,actf):
        super(NN3, self).__init__()
        self.width1 = width1
        self.width2 = width2
        self.depth1 = depth1
        self.depth2 = depth2
        self.bn = bn
        self.dp = dp
        self.dprate = dprate
        self.actf = actf
        self.in_N = in_N
        self.out_N = out_N
        self.stack = nn.ModuleList()
        self.stack.append(Unit3(in_N, width1[0],actf))
        if bn==1:
            self.stack.append(nn.BatchNorm1d(width1[0]))
        for i in range(1,depth1):
            self.stack.append(Unit3(width1[i-1], width1[i],actf))
        
        if dp==1:
            self.stack.append(nn.Dropout(p=dprate))
        if depth2==1:
            self.stack.append(Unit3(width1[i], width2[0],1)) 
        else:
            self.stack.append(Unit3(width1[i], width2[0],actf))    
            for i in range(1,depth2-1):
                self.stack.append(Unit3(width2[i-1], width2[i],actf))
            self.stack.append(Unit3(width2[depth2-2], width2[depth2-1],1)) 
            
    def forward(self, x):
        for i in range(len(self.stack)):
            x = self.stack[i](x)
        return x

activation=0
dropout=1
dropout_rate=0.53903
normalization=0
batch_size=345
layers1=10
layers2=0
neurons=44
learning_rate=0.00043
L1=[neurons]*layers1
L2=[neurons]*layers2+[8]
model_h = NN3(34,L1,layers1,L2,layers2+1, 8,normalization,dropout,dropout_rate,0)       
load=1
PATH="checkpoint/model-2784.pt"
if load==1:
    checkpoint = torch.load(PATH)
    model_h.load_state_dict(checkpoint['model_h_state_dict'])
    optimizer2 = optim.AdamW([{'params': model_h.parameters()}], lr=learning_rate) #Default=1e-4
    optimizer2.load_state_dict(checkpoint['optimizer2_state_dict']) 
model_h.eval()
xlo_test=np.load('xlo_test.npy')
ylo_test=np.load('ylo_test.npy')
pred_2h_star_test = model_h(torch.from_numpy(xlo_test).float())
acc_hi_0_test=torch.round(pred_2h_star_test).int()
acc_hi_1_test=torch.from_numpy(ylo_test).int()
Conf_test=torch.cat((acc_hi_0_test,acc_hi_1_test),dim=1)


feature_names =[r"$\mathrm{\Delta S_{mix}}$",r"$\mathrm{\Delta H_{mix}}$",r"$\mathrm{\Omega}$",r"$\mathrm{\eta}$","$\mathrm{k}_{\mathrm{1}}$","$\mathrm{\phi}$","$\mathrm{\delta}$","VEC","$\mathrm{PFP}_{\mathrm{FCC}}$","$\mathrm{PFP}_{\mathrm{BCC}}$","$\mathrm{PFP}_{\mathrm{HCP}}$","$\mathrm{PFP}_{\mathrm{Ordered\ BCC}}$","$\mathrm{PFP}_{\mathrm{Laves}}$","$\mathrm{PFP}_{\mathrm{Sigma}}$","$\mathrm{PSP}}$","$\sigma_{\mathrm{\Delta H_{mix}}}$","$\mathrm{K}}$","$\sigma_{\mathrm{K}}$","$\mathrm{T}_{\mathrm{m}}$","$\sigma_{\mathrm{T}_{\mathrm{m}}}$","$\sigma_{\mathrm{VEC}}$","$\mathrm{\chi}$","Atomic number","Group","Families","L quantum number","Miracle radius","Covalent radius","Zunger radius","MB electronegativity","Gordy electronegativity","Boiling point","Density","Specific heat"]
num=1
A=np.load('xlo_fold1.npy')
B=np.load('ylo_fold1.npy')
n=np.zeros([0,np.shape(A)[1]])
for i in range(np.shape(A)[0]):
    if B[i,0]==0 and B[i,1]==1 and B[i,2]==0 and B[i,3]==1 and B[i,4]==1 and B[i,5]==0 and B[i,6]==0 and B[i,7]==0: #A2-B2-Laves 3_phase 
        n=np.append(n,A[[i],:],axis=0)
Test=n
A=np.copy(Conf_test)
n=[]
m=[]
p=[]
for i in range(np.shape(A)[0]):
    if A[i,0+8]==0 and A[i,1+8]==1 and A[i,2+8]==0 and A[i,3+8]==1 and A[i,4+8]==1 and A[i,5+8]==0 and A[i,6+8]==0 and A[i,7+8]==0: #A2-B2-Laves 3_phase 
        if A[i,0]==0 and A[i,1]==1 and A[i,2]==0 and A[i,3]==1 and A[i,4]==1 and A[i,8]==0 and A[i,6]==0 and A[i,7]==0: #A2-B2-Laves 3_phase  
            m=np.append(m,i)
        else:
            n=np.append(n,i)      
        
B=np.load('xlo_test.npy')
C=np.zeros([0,np.shape(B)[1]])
Cy=np.zeros([0,8])
D=np.zeros([0,np.shape(B)[1]])
E=np.zeros([0,np.shape(B)[1]])
for i in range(np.shape(n)[0]):
    C=np.append(C,B[[int(n[i])],:],axis=0)
    Cy=np.append(Cy,A[[int(n[i])],0:8],axis=0)
for i in range(np.shape(m)[0]):
    D=np.append(D,B[[int(m[i])],:],axis=0)
for i in range(np.shape(p)[0]):
    E=np.append(E,B[[int(p[i])],:],axis=0)
VEC1=Test[:,[num]] 
VEC2=D[:,[num]] 
VEC3=C[:,[num]] 
VEC4=E[:,[num]] 

bin_range = (0, 1)
bin_size = 0.1
num_bins = int((bin_range[1] - bin_range[0]) / bin_size)
bins = np.arange(bin_range[0], bin_range[1] + bin_size, bin_size)



fig, axes = plt.subplots(1, 3,figsize =(18, 5),tight_layout = True)
axes[0].hist(VEC1, bins=bins,color='black') 
axes[0].set_yscale('log')
axes[0].set_ylabel('Count',fontsize=25, fontname='Times New Roman')
axes[0].set_xlabel(feature_names[num],fontsize=25, fontname='Times New Roman')
axes[0].tick_params(axis='x', labelsize=25)
axes[0].tick_params(axis='y', labelsize=25)
axes[0].set_xlim([0,1])
axes[0].set_ylim([0.1,20000])


axes[1].hist(VEC2, bins=bins,color='black') 
axes[1].set_yscale('log')
axes[1].set_ylabel('Count',fontsize=25, fontname='Times New Roman')
axes[1].set_xlabel(feature_names[num],fontsize=25, fontname='Times New Roman')
axes[1].tick_params(axis='x', labelsize=25)
axes[1].tick_params(axis='y', labelsize=25)
axes[1].set_xlim([0,1])
axes[1].set_ylim([0.1,20000])

axes[2].hist(VEC3, bins=bins,color='black') 
axes[2].set_yscale('log')
axes[2].set_ylabel('Count',fontsize=25, fontname='Times New Roman')
axes[2].set_xlabel(feature_names[num],fontsize=25, fontname='Times New Roman')
axes[2].tick_params(axis='x', labelsize=25)
axes[2].tick_params(axis='y', labelsize=25)
axes[2].set_xlim([0,1])
axes[2].set_ylim([0.1,20000])
plt.savefig('figure.png', dpi=600)